# Biohub Cell Tracking - Adaptive Soft Ellipsoid Classical Submission

Classical peak detection with adaptive per-sample caps, global-flow linking, and soft blob-size/ellipsoid ranking.


In [ ]:
from __future__ import annotations

import csv
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, label as connected_components
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max

try:
    from numcodecs import Blosc

    def decode_blosc(data: bytes) -> bytes:
        return Blosc().decode(data)

    BLOSC_BACKEND = 'numcodecs'
except Exception:
    try:
        import blosc2

        def decode_blosc(data: bytes) -> bytes:
            return blosc2.decompress(data)

        BLOSC_BACKEND = 'blosc2'
    except Exception as exc:
        raise ModuleNotFoundError(
            'Need numcodecs or blosc2 to decode Biohub Zarr chunks in Kaggle no-internet mode.'
        ) from exc

print('numpy', np.__version__)
print('pandas', pd.__version__)
print('blosc backend', BLOSC_BACKEND)


In [ ]:
# Configuration. These are deliberately conservative enough to run on Kaggle CPU.
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_CSV = Path('/kaggle/working/submission.csv')

# Data geometry from the competition page.
SCALE_ZYX = np.asarray([1.625, 0.40625, 0.40625], dtype=np.float32)

# Detector settings.
GAUSSIAN_SIGMA_ZYX = (0.7, 1.0, 1.0)
MIN_PEAK_DISTANCE = 3
LOW_THRESHOLD_QUANTILE = 50.0

# Hidden test has no GEFF metadata, so use an adaptive cap per frame.
# The two visible train samples estimate about 258 and 328 true nodes/frame.
TARGET_PEAKS_PER_FRAME = 300

# Link settings.
LINK_MAX_DISTANCE_UM = 7.0
ENABLE_GLOBAL_FLOW = True
FLOW_CONFIDENT_DISTANCE_UM = 4.0

# Global-flow linker.
ENABLE_GLOBAL_FLOW = True
FLOW_CONFIDENT_DISTANCE_UM = 4.0

# Soft ellipsoid/blob-size tuned ranking. These values come from the local blob survey.
ENABLE_BLOB_SIZE_RANKING = True
BLOB_TARGET_VOXELS = 928.0
BLOB_SIZE_SIGMA_VOXELS = 350.0
BLOB_SIZE_PENALTY_WEIGHT = 1.0
BLOB_RANKING_OVERSAMPLE_FACTOR = 3.0
BLOB_ALPHA = 0.35
BLOB_BACKGROUND_PERCENTILE = 20.0
BLOB_CROP_RADIUS_ZYX = (4, 8, 8)

# Adaptive per-sample detection cap from a cheap raw-intensity prescan.
ENABLE_ADAPTIVE_DETECTION = True
ADAPTIVE_PRESCAN_FRAMES = 5
ADAPTIVE_MIN_PEAKS_PER_FRAME = 180
ADAPTIVE_MAX_PEAKS_PER_FRAME = 850
ADAPTIVE_MEAN_SLOPE = 0.65
ADAPTIVE_MEAN_INTERCEPT = 130.0
ADAPTIVE_HIGH_BACKGROUND_P50 = 1200.0
ADAPTIVE_HIGH_BACKGROUND_CAP = 330


In [ ]:
def find_test_zarrs(input_root: Path) -> list[Path]:
    known = input_root / 'competitions' / 'biohub-cell-tracking-during-development' / 'test'
    if known.exists():
        return sorted(known.glob('*.zarr'))

    # Fallback for unexpected Kaggle input nesting. This is slower, so avoid it when possible.
    for test_dir in input_root.glob('**/test'):
        zarrs = sorted(test_dir.glob('*.zarr'))
        if zarrs:
            return zarrs
    return sorted(input_root.rglob('*.zarr'))


test_zarrs = find_test_zarrs(INPUT_ROOT)
print(f'found {len(test_zarrs)} zarr test samples')
for path in test_zarrs[:10]:
    print(path)
if len(test_zarrs) > 10:
    print(f'... {len(test_zarrs) - 10} more samples')
if not test_zarrs:
    raise FileNotFoundError('No .zarr samples found under /kaggle/input')


In [ ]:
class SimpleZarr3Array:
    def __init__(self, array_path: Path):
        self.array_path = Path(array_path)
        with (self.array_path / 'zarr.json').open(encoding='utf-8') as f:
            self.meta = json.load(f)

        self.shape = tuple(int(v) for v in self.meta['shape'])
        chunk_grid = self.meta['chunk_grid']['configuration']
        self.chunk_shape = tuple(int(v) for v in chunk_grid['chunk_shape'])
        self.dtype = self._dtype_from_meta(self.meta.get('data_type') or self.meta.get('dtype'))

    @staticmethod
    def _dtype_from_meta(data_type) -> np.dtype:
        text = str(data_type).lower()
        mapping = {
            'uint8': 'u1', 'uint16': '<u2', 'uint32': '<u4', 'uint64': '<u8',
            'int8': 'i1', 'int16': '<i2', 'int32': '<i4', 'int64': '<i8',
            'float32': '<f4', 'float64': '<f8',
        }
        if text not in mapping:
            raise ValueError(f'Unsupported Zarr dtype: {data_type}')
        return np.dtype(mapping[text])

    def _read_chunk(self, chunk_indices: tuple[int, int, int, int]) -> np.ndarray:
        chunk_path = (
            self.array_path / 'c' / str(chunk_indices[0]) / str(chunk_indices[1]) /
            str(chunk_indices[2]) / str(chunk_indices[3])
        )
        if not chunk_path.exists():
            return np.zeros(self.chunk_shape, dtype=self.dtype)
        decoded = decode_blosc(chunk_path.read_bytes())
        return np.frombuffer(decoded, dtype=self.dtype).reshape(self.chunk_shape)

    def __getitem__(self, item):
        # This notebook only needs whole timepoint reads: image[t, :, :, :].
        if not isinstance(item, tuple) or len(item) != 4 or not isinstance(item[0], (int, np.integer)):
            raise NotImplementedError('SimpleZarr3Array supports image[t, :, :, :] reads only')
        t = int(item[0])
        z_slice, y_slice, x_slice = item[1], item[2], item[3]
        if z_slice != slice(None) or y_slice != slice(None) or x_slice != slice(None):
            raise NotImplementedError('SimpleZarr3Array supports full Z/Y/X timepoint reads only')
        chunk = self._read_chunk((t, 0, 0, 0))
        return chunk[0, : self.shape[1], : self.shape[2], : self.shape[3]]


def open_zarr_array(zarr_path: Path, array_path: str = '0') -> SimpleZarr3Array:
    return SimpleZarr3Array(Path(zarr_path) / array_path)


In [ ]:
def estimate_detection_settings(image: SimpleZarr3Array) -> tuple[float, int]:
    if not ENABLE_ADAPTIVE_DETECTION:
        return LOW_THRESHOLD_QUANTILE, TARGET_PEAKS_PER_FRAME

    frame_count = int(image.shape[0])
    prescan_count = max(1, min(ADAPTIVE_PRESCAN_FRAMES, frame_count))
    frame_indices = np.linspace(0, frame_count - 1, prescan_count, dtype=int)
    stats = []
    for t in frame_indices:
        volume = np.asarray(image[int(t), :, :, :])
        positive = volume[volume > 0]
        if len(positive) == 0:
            continue
        stats.append((
            float(np.percentile(positive, 50)),
            float(np.percentile(positive, 99)),
            float(np.mean(positive)),
        ))

    if not stats:
        return LOW_THRESHOLD_QUANTILE, TARGET_PEAKS_PER_FRAME

    med_p50, med_p99, med_mean = np.median(np.asarray(stats, dtype=np.float32), axis=0)
    adaptive_cap = int(round(ADAPTIVE_MEAN_INTERCEPT + ADAPTIVE_MEAN_SLOPE * float(med_mean)))
    adaptive_cap = int(np.clip(adaptive_cap, ADAPTIVE_MIN_PEAKS_PER_FRAME, ADAPTIVE_MAX_PEAKS_PER_FRAME))
    if med_p50 >= ADAPTIVE_HIGH_BACKGROUND_P50 and (med_p99 - med_p50) <= 1400:
        adaptive_cap = min(adaptive_cap, ADAPTIVE_HIGH_BACKGROUND_CAP)

    print(
        'adaptive detection: '
        f'p50={med_p50:.1f}, p99={med_p99:.1f}, mean={med_mean:.1f}, '
        f'threshold_q={LOW_THRESHOLD_QUANTILE:.1f}, max_peaks={adaptive_cap}'
    )
    return LOW_THRESHOLD_QUANTILE, adaptive_cap


def local_blob_voxel_count(smoothed: np.ndarray, peak_zyx: np.ndarray) -> int:
    center = peak_zyx.astype(np.int64)
    radius = np.asarray(BLOB_CROP_RADIUS_ZYX, dtype=np.int64)
    lo = np.maximum(center - radius, 0)
    hi = np.minimum(center + radius + 1, np.asarray(smoothed.shape, dtype=np.int64))
    crop = smoothed[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    local_peak = center - lo
    peak_value = float(crop[tuple(local_peak)])
    background = float(np.percentile(crop, BLOB_BACKGROUND_PERCENTILE))
    threshold = background + BLOB_ALPHA * (peak_value - background)
    mask = crop >= threshold
    labels, label_count = connected_components(mask)
    if label_count == 0:
        return 0
    component_id = int(labels[tuple(local_peak)])
    if component_id == 0:
        return 0
    return int(np.count_nonzero(labels == component_id))


def rank_peaks_by_soft_blob_size(
    smoothed: np.ndarray,
    peaks: np.ndarray,
    intensities: np.ndarray,
    target_peaks_per_frame: int,
) -> tuple[np.ndarray, np.ndarray]:
    log_intensities = np.log1p(np.maximum(intensities.astype(np.float64), 0.0))
    intensity_scale = float(np.std(log_intensities)) or 1.0
    candidates = []
    for peak, intensity in zip(peaks, intensities):
        voxel_count = local_blob_voxel_count(smoothed, peak.astype(np.int64))
        size_z = (float(voxel_count) - BLOB_TARGET_VOXELS) / max(float(BLOB_SIZE_SIGMA_VOXELS), 1.0)
        size_penalty = BLOB_SIZE_PENALTY_WEIGHT * size_z * size_z
        score = np.log1p(max(float(intensity), 0.0)) / intensity_scale - size_penalty
        candidates.append((float(score), peak, float(intensity)))
    if not candidates:
        return np.zeros((0, 3), dtype=np.int64), np.zeros((0,), dtype=np.float32)
    candidates.sort(key=lambda item: item[0], reverse=True)
    selected = candidates[:target_peaks_per_frame]
    return (
        np.asarray([item[1] for item in selected]),
        np.asarray([item[2] for item in selected], dtype=np.float32),
    )


def detect_frame(
    volume: np.ndarray,
    threshold_quantile: float,
    target_peaks_per_frame: int,
) -> np.ndarray:
    smoothed = gaussian_filter(volume.astype(np.float32), sigma=GAUSSIAN_SIGMA_ZYX)
    positive = smoothed[smoothed > 0]
    if len(positive) == 0:
        return np.zeros((0, 4), dtype=np.float32)

    threshold = float(np.percentile(positive, threshold_quantile))
    num_peaks = target_peaks_per_frame
    if ENABLE_BLOB_SIZE_RANKING:
        num_peaks = max(target_peaks_per_frame, int(round(target_peaks_per_frame * BLOB_RANKING_OVERSAMPLE_FACTOR)))
    peaks = peak_local_max(
        smoothed,
        min_distance=MIN_PEAK_DISTANCE,
        threshold_abs=threshold,
        exclude_border=False,
        num_peaks=num_peaks,
    )
    if len(peaks) == 0:
        return np.zeros((0, 4), dtype=np.float32)

    intensities = smoothed[peaks[:, 0], peaks[:, 1], peaks[:, 2]]
    order = np.argsort(-intensities)
    peaks = peaks[order]
    intensities = intensities[order]
    if ENABLE_BLOB_SIZE_RANKING:
        peaks, intensities = rank_peaks_by_soft_blob_size(smoothed, peaks, intensities, target_peaks_per_frame)
    else:
        peaks = peaks[:target_peaks_per_frame]
        intensities = intensities[:target_peaks_per_frame]
    return np.column_stack([peaks, intensities]).astype(np.float32)


def build_nodes(zarr_path: Path, array_path: str = '0') -> list[dict]:
    image = open_zarr_array(zarr_path, array_path)
    threshold_quantile, target_peaks_per_frame = estimate_detection_settings(image)
    nodes = []
    next_id = 1
    for t in range(int(image.shape[0])):
        peaks = detect_frame(np.asarray(image[t, :, :, :]), threshold_quantile, target_peaks_per_frame)
        if t % 10 == 0 or t == int(image.shape[0]) - 1:
            print(f'{zarr_path.stem} t={t:03d}: {len(peaks)} peaks')
        for z, y, x, score in peaks:
            nodes.append({
                'node_id': next_id,
                't': int(t),
                'z': int(round(float(z))),
                'y': int(round(float(y))),
                'x': int(round(float(x))),
                'score': float(score),
            })
            next_id += 1
    return nodes


def solve_frame_links(
    prev_xyz: np.ndarray,
    curr_xyz: np.ndarray,
    max_distance_um: float,
    flow_um: np.ndarray | None = None,
) -> list[tuple[int, int, float]]:
    if flow_um is None:
        flow_um = np.zeros(3, dtype=np.float32)
    predicted_xyz = prev_xyz + flow_um
    residual = np.linalg.norm(predicted_xyz[:, None, :] - curr_xyz[None, :, :], axis=2)
    cost = residual.copy()
    cost[cost > max_distance_um] = 1e6
    row_ind, col_ind = linear_sum_assignment(cost)

    links = []
    for r, c in zip(row_ind, col_ind):
        residual_distance = float(residual[r, c])
        if residual_distance <= max_distance_um:
            links.append((int(r), int(c), residual_distance))
    return links


def estimate_global_flow_um(prev_xyz: np.ndarray, curr_xyz: np.ndarray) -> np.ndarray:
    initial_links = solve_frame_links(prev_xyz, curr_xyz, LINK_MAX_DISTANCE_UM)
    displacements = []
    for r, c, _ in initial_links:
        actual_distance = float(np.linalg.norm(curr_xyz[c] - prev_xyz[r]))
        if actual_distance <= FLOW_CONFIDENT_DISTANCE_UM:
            displacements.append(curr_xyz[c] - prev_xyz[r])
    if not displacements:
        return np.zeros(3, dtype=np.float32)
    return np.median(np.asarray(displacements, dtype=np.float32), axis=0).astype(np.float32)


def link_nodes(nodes: list[dict]) -> list[tuple[int, int]]:
    by_t = defaultdict(list)
    for node in nodes:
        by_t[node['t']].append(node)

    edges = []
    times = sorted(by_t)
    for t0, t1 in zip(times[:-1], times[1:]):
        if t1 != t0 + 1:
            continue
        prev = by_t[t0]
        curr = by_t[t1]
        if not prev or not curr:
            continue

        prev_xyz = np.asarray([[n['z'], n['y'], n['x']] for n in prev], dtype=np.float32) * SCALE_ZYX
        curr_xyz = np.asarray([[n['z'], n['y'], n['x']] for n in curr], dtype=np.float32) * SCALE_ZYX
        flow_um = estimate_global_flow_um(prev_xyz, curr_xyz) if ENABLE_GLOBAL_FLOW else np.zeros(3, dtype=np.float32)
        links = solve_frame_links(prev_xyz, curr_xyz, LINK_MAX_DISTANCE_UM, flow_um=flow_um)
        for r, c, _ in links:
            edges.append((prev[r]['node_id'], curr[c]['node_id']))
    return edges

In [ ]:
rows = []

for zarr_path in test_zarrs:
    dataset = zarr_path.stem
    print(f'processing {dataset}')
    nodes = build_nodes(zarr_path)
    edges = link_nodes(nodes)
    print(f'{dataset}: nodes={len(nodes)}, edges={len(edges)}')

    for node in nodes:
        rows.append({
            'dataset': dataset,
            'row_type': 'node',
            'node_id': node['node_id'],
            't': node['t'],
            'z': node['z'],
            'y': node['y'],
            'x': node['x'],
            'source_id': -1,
            'target_id': -1,
        })

    for source_id, target_id in edges:
        rows.append({
            'dataset': dataset,
            'row_type': 'edge',
            'node_id': -1,
            't': -1,
            'z': -1,
            'y': -1,
            'x': -1,
            'source_id': source_id,
            'target_id': target_id,
        })

In [ ]:
submission_columns = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
submission = pd.DataFrame(rows)
submission.insert(0, 'id', np.arange(len(submission), dtype=np.int64))
submission = submission[submission_columns]
submission.to_csv(OUTPUT_CSV, index=False)

print(f'wrote {OUTPUT_CSV}')
print(submission.shape)
print(submission.head())
print(submission.tail())